In [1]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")

from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/alarm_history_dump.parquet")
graph_repo = AlarmGraphRepository(os.getenv("HISTORY_DB_PATH"))

In [2]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationHistory

SimpleTimeCorrelationHistory.train(lazy_frame, graph_repo)

Processando nós: 100%|██████████| 912/912 [05:25<00:00,  2.80nó/s] 


In [3]:
from src.postprocess.enumerate_incidents import EnumerateIncidents

EnumerateIncidents.enumerate_data(graph_repo)

graph_repo.preview_nodes()

alert_id,incident,alert_type,start_time,end_time,node_id
str,i32,str,datetime[μs],datetime[μs],str
"""65a5d542-bbdf-4481-a3a8-7c69a5…",155988,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-12 05:36:12.939274,null,"""0735063a-6a73-4b88-9c87-36c374…"
"""ed999673-98ad-4950-b566-c7c4a2…",156470,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-15 16:40:49.653298,null,"""0735063a-6a73-4b88-9c87-36c374…"
"""66e41ca3-5d34-4d01-b8b5-e4cea0…",157879,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-26 12:20:56.965146,null,"""0735063a-6a73-4b88-9c87-36c374…"
"""2c5009fe-e6a9-4ccf-a334-0c94a6…",158228,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-28 22:05:45.376166,null,"""0735063a-6a73-4b88-9c87-36c374…"
"""6141e7d2-a0a0-4e5e-941c-64b957…",154610,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-02 11:41:28.805688,null,"""0735063a-6a73-4b88-9c87-36c374…"
…,…,…,…,…,…
"""f5289c2b-fedb-4180-b1e2-ec9089…",156575,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-16 11:05:59.971868,null,"""0735063a-6a73-4b88-9c87-36c374…"
"""f2c103f0-03eb-4fc9-bac1-dd763b…",155525,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-08 23:31:10.052212,null,"""0735063a-6a73-4b88-9c87-36c374…"
"""35f785bc-f6a4-4a9d-b695-4df567…",157977,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-27 04:30:38.374687,null,"""0735063a-6a73-4b88-9c87-36c374…"


In [4]:
from src.utils.node_summary import node_summary
from src.repository.aggregate_results_repository import AggregateResultsRepository

summary, general_metrics = node_summary(graph_repo)

results_repo = AggregateResultsRepository(filename="simple_time_corr_history")
results_repo.save(summary)
results_repo.load()

✅ Nova versão salva com sucesso em: data/results/simple_time_corr_history_20260530_140019.csv
📖 Carregando a versão mais recente encontrada: simple_time_corr_history_20260530_140019.csv


Node ID,Total de Alarmes,Total de Correlações,Total de Incidentes,Média de Alarmes por Incidente,Densidade
str,i64,i64,i64,f64,f64
"""a5126d4f-0bd1-4023-ae9d-d2c1e9…",232712,8584164,4090,56.8978,0.000317
"""7510d37a-4c5f-40da-b987-3977c2…",32148,119403,4073,7.892954,0.000231
"""8bf0d4d2-1cd3-4341-9b79-794833…",31766,116570,4065,7.814514,0.000231
"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…",32023,119030,4054,7.899112,0.000232
"""75577cb5-6565-42da-a042-d9603e…",27157,83723,4041,6.720366,0.000227
…,…,…,…,…,…
"""e9488358-73d3-4612-a413-62ac2b…",714,0,1,714.0,0.0
"""0b466dd2-3057-4816-9379-ac1e7e…",12,0,1,12.0,0.0
"""0ea468f1-9c1d-4386-9ea8-8ba975…",711,0,1,711.0,0.0


In [5]:
graph_repo.close()